# ModelV3 inference on a real MACSIMA image

Apply the v3 inversion model -- trained **only on synthetic data** -- to a real myocarditis MACSIMA
image. We predict the cell + nucleus instance segmentation (both from Omnipose flows) and the
generative parameters, then **render synthetic samples from those learned parameters**.

Uses `runs/run12/model_backsub_final.pt` -- the newest model (step 276000): per-marker normalisation + expression-strength, nucleus flows, background-only markers, the full cell/structure DENSITY spectrum (which it also PREDICTS), and trained in the **MACSIMA background-subtracted domain** (empty regions are near-zero speckle, not a pedestal) so it matches the staged `*_backsub.ome.tif` inputs. Runs in the `synmac` env (torch + scipy + tifffile + zarr + skimage). The
model is out-of-distribution on real data, so this is exploratory.

Runs on the **staged, background-subtracted** MACSIMA output (the real pipeline domain); the generator simulates that background subtraction so empty regions match and are not oversegmented.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))
import numpy as np, torch, tifffile, zarr
from scipy import ndimage as ndi
from skimage.segmentation import find_boundaries
import matplotlib.pyplot as plt

import config as cfg, flexible_types as ft
from model import normalize_markers
from model_v3 import ModelV3
import gen_dataset as gd
from parameter import (CellGeometry, CellType, MarkerPanel, P, dapi_marker,
                       narrow, from_u, Detector, TissueGeometry)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CKPT   = "../runs/run12/model_backsub_final.pt"   # newest: background-subtracted model (30 rounds, step 276000)
# staged MACSIMA output (background-subtracted + clipped at 0) -- the domain the model is
# now trained on. DAPI (index 0) is the registration target and is NOT background-subtracted.
IMAGE  = ("/mnt/Transfer/Myocarditis26/20260130_AGBorst_Myocarditis_Slide2_260216_420907_R1_A1_ROI2_staged/1/rack-01-well-A01-roi-002-exp-2/background/rack-01-well-A01-roi-002-exp-2_backsub.ome.tif")

ck = torch.load(CKPT, map_location=DEVICE, weights_only=False)
model = ModelV3(n_global=ck["cfg"]["n_global"], n_marker_noise=4,
                K=ck["cfg"]["k_slots"], T=3).to(DEVICE).eval()
model.load_state_dict(ck["model"])
print(f"loaded checkpoint at step {ck['step']}  |  device {DEVICE}")


def flow_to_instances(fy, fx, mask, n_iter=150, step=1.0, thresh_frac=0.02, min_size=12):
    """Follow an Omnipose flow field to its sinks and cluster endpoints into instances."""
    H, W = mask.shape
    ys, xs = np.nonzero(mask)
    if ys.size == 0:
        return np.zeros((H, W), np.int32)
    py, px = ys.astype(np.float32).copy(), xs.astype(np.float32).copy()
    def samp(F, py, px):
        y0 = np.clip(np.floor(py).astype(int), 0, H-1); x0 = np.clip(np.floor(px).astype(int), 0, W-1)
        y1 = np.clip(y0+1, 0, H-1); x1 = np.clip(x0+1, 0, W-1); wy = py-y0; wx = px-x0
        return (F[y0,x0]*(1-wy)*(1-wx) + F[y0,x1]*(1-wy)*wx + F[y1,x0]*wy*(1-wx) + F[y1,x1]*wy*wx)
    for _ in range(n_iter):
        py = np.clip(py + step*samp(fy, py, px), 0, H-1)
        px = np.clip(px + step*samp(fx, py, px), 0, W-1)
    ey = np.clip(np.round(py).astype(int), 0, H-1); ex = np.clip(np.round(px).astype(int), 0, W-1)
    hist = np.zeros((H, W), np.float32); np.add.at(hist, (ey, ex), 1.0)
    hist = ndi.gaussian_filter(hist, 1.0)
    basins, _ = ndi.label(hist > hist.max()*thresh_frac)
    inst = np.zeros((H, W), np.int32); inst[ys, xs] = basins[ey, ex]
    for lab in np.unique(inst):
        if lab and (inst == lab).sum() < min_size: inst[inst == lab] = 0
    u = np.unique(inst); remap = {v: i for i, v in enumerate(u)}
    return np.vectorize(remap.get)(inst).astype(np.int32)

## 1. Load a crop of the real image, rescaled to the training resolution

72 markers at 0.17 um/px; the model trained at 0.325 um/px, so a 1024 px crop downsampled 2x -> 512
px (~0.34 um/px).

In [ ]:
R0, C0, S = 10239, 28006, 1024    # a tissue-rich region of the staged slide
store = tifffile.imread(IMAGE, aszarr=True)
z = zarr.open(store, mode="r")
full = z["0"] if hasattr(z, "keys") else z
crop = np.asarray(full[:, R0:R0+S, C0:C0+S])          # [48, 1024, 1024] uint16 (backsub)
ds = ndi.zoom(crop.astype(np.float32), (1, 0.5, 0.5), order=1)   # -> [72, 512, 512]
M = ds.shape[0]
print("real crop", crop.shape, "-> downsampled", ds.shape)

## 2. Per-marker normalisation + inference

Each marker is normalised to [0,1] and a per-marker **expression strength** is computed (the same
`normalize_markers` the model trains with), so a 10-100x brighter DAPI no longer dominates and the
model knows each channel's relative brightness.

In [ ]:
norm, strength = normalize_markers(np.moveaxis(ds, 0, -1))     # [H,W,M], [M]
tile = torch.from_numpy(norm).permute(2, 0, 1)[None]
strn = torch.from_numpy(strength)[None]
with torch.no_grad():
    out = model(tile.to(DEVICE), present=torch.ones(1, M, dtype=torch.bool, device=DEVICE),
                strength=strn.to(DEVICE))

nuc      = torch.sigmoid(out["dense"]["nucleus"][0, 0]).cpu().numpy()
cell     = torch.sigmoid(out["dense"]["cell"][0, 0]).cpu().numpy()
flow     = out["dense"]["flow"][0].cpu().numpy()       # cell flows (fy, fx)
nuc_flow = out["dense"]["nuc_flow"][0].cpu().numpy()   # nucleus flows (fy, fx)
so = out["slots"][-1]
exist = so["existence"][0].sigmoid().cpu().numpy()
present = [k for k in range(len(exist)) if exist[k] > 0.5]
print(f"inferred {len(present)} cell types  |  strength range "
      f"[{strength.min():.2f}, {strength.max():.2f}]  (low = background-like channel)")

# newest model also predicts the global tissue DENSITY (prepended at global indices 0-3)
g_u = out["global_mu"][0].sigmoid().cpu().numpy()         # normalised [0,1]
_tg = TissueGeometry()
pred_min_dist = from_u(_tg.MIN_DIST, float(g_u[0]))      # cell spacing, voxels (small = dense)
pred_cover    = from_u(_tg.COVER,    float(g_u[1]))      # tissue coverage (1 = no background)
pred_grow     = from_u(_tg.GROW,     float(g_u[2]))      # confluence
pred_struct   = float(g_u[3])                            # structure territory fraction
print(f"predicted density: MIN_DIST~{pred_min_dist:.1f} vox  COVER~{pred_cover:.2f}  "
      f"GROW~{pred_grow:.2f}  structure_frac~{pred_struct:.2f}")

## 3. Sample crops from the original image

In [ ]:
def nrm(a, p=99.5):
    a = a.astype(np.float32); hi = np.percentile(a, p)
    return np.clip(a / (hi + 1e-6), 0, 1)

SHOW = [0, 24, 7, 27]                                   # 0 = DAPI / nuclear stain
real_rgb = np.stack([nrm(ds[24]), nrm(ds[7]), nrm(ds[0])], -1)   # R, G, B(=DAPI)

fig, ax = plt.subplots(1, 5, figsize=(18, 4))
for i, c in enumerate(SHOW):
    ax[i].imshow(nrm(ds[c]), cmap="gray")
    ax[i].set_title(f"channel {c}" + (" (DAPI)" if c == 0 else ""))
ax[4].imshow(real_rgb); ax[4].set_title("RGB composite 24/7/0")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 4. Predicted segmentation on the image (nuclei + cells, both from flows)

Nuclei are now separated via a dedicated **nucleus flow field** (not a semantic blob), so touching
nuclei split into distinct instances. Cells come from the cell flow field.

In [ ]:
nuc_inst  = flow_to_instances(nuc_flow[0], nuc_flow[1], nuc > 0.5)
cell_inst = flow_to_instances(flow[0], flow[1], cell > 0.5)
n_nuc, n_cell = len(set(np.unique(nuc_inst)) - {0}), len(set(np.unique(cell_inst)) - {0})

ov = real_rgb.copy(); ov[find_boundaries(cell_inst, mode="thick")] = [1, 1, 0]
fig, ax = plt.subplots(1, 4, figsize=(16, 4.3))
ax[0].imshow(nuc, cmap="magma"); ax[0].set_title("pred nucleus prob")
ax[1].imshow(nuc_inst, cmap="nipy_spectral"); ax[1].set_title(f"nucleus instances (flows): {n_nuc}")
ax[2].imshow(cell_inst, cmap="nipy_spectral"); ax[2].set_title(f"cell instances (flows): {n_cell}")
ax[3].imshow(ov); ax[3].set_title("cell boundaries on image")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 5. Samples generated from the learned parameters

Reconstruct a renderable *world* from the predicted parameters (per-marker `mu`/texture, each
inferred cell type's geometry + marker expression) and render synthetic tiles -- the tissue as the
model understood it. Subset of markers for speed; placement is re-sampled so it won't match the crop.

In [ ]:
mo = out["marker"]
mu_u    = mo["mu"][0].sigmoid().cpu().numpy()
noise_u = mo["noise_mu"][0].sigmoid().cpu().numpy()
pos  = so["expression"]["positive"][0].sigmoid().cpu().numpy()
lvl  = so["expression"]["level_mu"][0].sigmoid().cpu().numpy()
geom = so["geometry"]["mu"][0].sigmoid().cpu().numpy()

M = mu_u.shape[0]                                   # channels in THIS image (panel varies per image)
_strength = (pos[present] * lvl[present]).max(0) if len(present) else np.zeros(M)
SEL = [int(c) for c in np.argsort(_strength)[::-1][:6]]   # the 6 most-expressed channels (always in range)
b_min, b_max = ft.expression_range(); LGL, LGH = np.log10(b_min), np.log10(b_max)
GLO, GHI = ft._geometry_bounds()

def unnorm_geom(u):
    g = {}
    for i, k in enumerate(ft.GEOM_KEYS):
        g[k] = (float(np.exp(u[i]*(np.log(GHI['RADIUS'])-np.log(GLO['RADIUS']))+np.log(GLO['RADIUS'])))
                if k == 'RADIUS' else GLO[k] + u[i]*(GHI[k]-GLO[k]))
    return CellGeometry(**{k: float(v) for k, v in g.items()})

markers = {"DAPI": dapi_marker(name="DAPI")}
for c in SEL:
    mu_v = cfg.MU_LO + float(mu_u[c]) * (cfg.MU_HI - cfg.MU_LO)
    mk = ft.build_marker(np.random.default_rng(c), name=f"M{c}", dye=cfg.DYES[c % 3],
                         mu=mu_v, noise_kind_idx=[1])
    comp = mk.noise_components[0]
    for j, key in enumerate(["scale", "clust", "fill", "s"]):
        p = getattr(comp, key, None)
        if isinstance(p, P):
            setattr(comp, key, narrow(p, p.lo, p.hi, from_u(p, float(noise_u[c, j]))))
    markers[f"M{c}"] = mk
panel = MarkerPanel(Markers=markers)

types = {}
for i, k in enumerate(present):
    expr = {f"M{c}": P(0., b_max, .05, float(10**(LGL + float(lvl[k, c])*(LGH-LGL))), "e")
            for c in SEL if pos[k, c] > 0.5}
    types[f"T{i}"] = CellType(name=f"T{i}", Geometry=unnorm_geom(geom[k]), Expression=expr)

arch, kinds = ft._flexible_architecture(np.random.default_rng(1))
contexts = ["background"] + list(dict.fromkeys(kinds)); K = len(present)
comps = dict(contexts=contexts, pi=np.ones((len(contexts), K)) / K,
             allowed=np.ones((len(contexts), K), bool),
             struct_to_ctx=[contexts.index(kd) for kd in kinds], type_order=list(types))
# render the samples at the DENSITY the model inferred from the real crop
rtg = TissueGeometry()
rtg.MIN_DIST = narrow(rtg.MIN_DIST, rtg.MIN_DIST.lo, rtg.MIN_DIST.hi, pred_min_dist)
rtg.COVER    = narrow(rtg.COVER, rtg.COVER.lo, rtg.COVER.hi, pred_cover)
rtg.GROW     = narrow(rtg.GROW, rtg.GROW.lo, rtg.GROW.hi, pred_grow)
world = dict(PANEL=panel, TYPES=types, TG=rtg, DET=Detector(**cfg.DETECTOR),
             ARCH=arch, CELL=CellGeometry(), FRACTIONS=list(np.ones(K) / K), COMPOSITIONS=comps)
names = list(panel.Markers)
print(f"reconstructed panel {names}  |  {K} cell types")

In [ ]:
_mk = [m for m in names if m != "DAPI"]
r_ch = _mk[0] if len(_mk) > 0 else "DAPI"
g_ch = _mk[1] if len(_mk) > 1 else r_ch
ri = [names.index(r_ch), names.index(g_ch), names.index("DAPI")]   # two markers + DAPI(blue)
samples = [gd.render_frame(world, seed=s, pool_bank=cfg.POOL_BANK, with_artifacts=False)[0]
           for s in (1, 2, 3, 4)]
fig, ax = plt.subplots(1, 5, figsize=(18, 4))
ax[0].imshow(real_rgb); ax[0].set_title("REAL composite (reference)")
for j, s in enumerate(samples):
    rgb = np.stack([nrm(s[..., ri[0]]), nrm(s[..., ri[1]]), nrm(s[..., ri[2]])], -1)
    ax[j+1].imshow(rgb); ax[j+1].set_title(f"generated #{j+1}")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

### Notes
- Zero-shot transfer: trained only on synthetic data. Nuclei and cells are now both instance-
  segmented from Omnipose flows (touching nuclei separate).
- The per-marker **strength** is a model INPUT (wired into the Stem and Trunk): since each image
  channel is normalised to [0,1] and thus brightness-invariant, strength is how the model tells a
  strongly-expressed channel from a dim / background-like one (low strength). It USES strength -- it
  does not ignore it.
- Generated samples re-sample placement and reconstruct a marker subset, so they show the inferred
  *biology*, not a pixel reconstruction. No ground truth exists on real data.
- The generated samples are now rendered at the model's **predicted** cell/structure density.
- Input domain: the staged **background-subtracted** pipeline output (empty regions are near-zero
  with sparse speckle); the generator now simulates MACSIMA backsub so synthetic and real match.
- Checkpoint: `runs/run12/model_backsub_final.pt` -- newest model, step 276000, trained in the background-subtracted domain (`step` printed on load).